# Tutorial 2: Person Re-Identification (ReID) Across Cameras

**Pipeline Stage:** Matching the same person across different camera views

---

## Overview

This tutorial covers the **second stage** of the multi-camera human tracking pipeline:
using **appearance-based embeddings** to determine which person detected in one camera
is the same person seen in another camera.

### Why ReID BEFORE Triangulation?

This is the critical insight: **triangulation requires knowing which detections across
cameras correspond to the same person.** If Camera 1 sees 5 people and Camera 3 sees
5 people, triangulation needs to know that "Person 2 in Cam1" = "Person 4 in Cam3"
before it can combine their 2D keypoints into 3D coordinates.

Without ReID, you'd be triangulating random pairings of different people — garbage in,
garbage out.

### What You Will Learn

1. **Installation** — BoxMOT, torchreid, OSNet weights, and dependencies
2. **Person Cropping** — Extracting person bounding-box crops from video + pose data
3. **Shirt/Torso Cropping** — Focused torso crops for more robust matching
4. **ReID Embeddings** — Extracting appearance feature vectors with OSNet
5. **Cosine Similarity** — Measuring appearance similarity between crops
6. **Per-Frame Cross-Camera Matching** — Identifying the same person across views for each frame
7. **Building the Identity Map** — Creating cross-camera correspondence tables for triangulation

### Pipeline Context

```
┌─────────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐
│  Tutorial 1          │ ──► │  Tutorial 2 (HERE)    │ ──► │  Tutorial 3          │
│  YOLO Pose (2D)     │     │  Person ReID          │     │  3D Triangulation    │
│  per-camera          │     │  cross-camera match   │     │  multi-camera fusion │
└─────────────────────┘     └──────────────────────┘     └─────────────────────┘
                                      │
                                      ▼
                         For each frame, answers:
                         "Detection 2 in CAM1 =
                          Detection 0 in CAM3 =
                          Detection 1 in CAM5"
```

### Prerequisites

- Completed Tutorial 1 (2D pose results with bounding boxes for all cameras)
- SLEAP analysis H5 files with tracked poses
- Original synchronized video files for cropping
- GPU recommended for embedding extraction

---

## Part 1: Installation

| Package | Purpose | Install |
|---|---|---|
| `boxmot` | Multi-object tracking with ReID encoders | `pip install boxmot` |
| `torch` / `torchvision` | Deep learning framework | `pip install torch torchvision` |
| `scikit-learn` | Cosine similarity computation | `pip install scikit-learn` |
| `opencv-python` | Video reading, image cropping | `pip install opencv-python` |
| `h5py` | Reading SLEAP pose H5 files | `pip install h5py` |
| `Pillow` | Image loading and transforms | `pip install Pillow` |
| `matplotlib` | Visualization | `pip install matplotlib` |

### ReID Model: OSNet

We use **OSNet (Omni-Scale Network)** — a lightweight model designed specifically for
person re-identification. The `osnet_x0_25` variant is:
- Very fast (~1ms per crop on GPU)
- Produces 512-dimensional appearance embeddings
- Pre-trained on ImageNet, fine-tunable on ReID datasets

### ReID Weights

Download pre-trained weights:
- `osnet_x0_25_imagenet.pth` — general-purpose, works well out of the box
- Available from the [deep-person-reid](https://github.com/KaiyangZhou/deep-person-reid) repository

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# Uncomment and run if not yet installed:

# !pip install boxmot torch torchvision scikit-learn opencv-python h5py Pillow matplotlib tqdm

In [ ]:
# ============================================================
# STEP 2: Verify installation and GPU
# ============================================================
import torch
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")

import cv2
print(f"OpenCV:     {cv2.__version__}")

import h5py
print(f"h5py:       {h5py.__version__}")

from sklearn.metrics.pairwise import cosine_similarity
print(f"sklearn:    available")

try:
    from boxmot.trackers.strongsort.strongsort import StrongSort
    print(f"BoxMOT:     available (StrongSort)")
except ImportError:
    print("WARNING: BoxMOT not installed. Install with: pip install boxmot")

---

## Part 2: Understanding the ReID Pipeline

### The Big Picture — Per-Frame Cross-Camera Matching

For **each frame** in our synchronized multi-camera video, we need to:

```
Frame N:
  CAM1 detects: [Person A, Person B, Person C]
  CAM2 detects: [Person X, Person Y]
  CAM3 detects: [Person P, Person Q, Person R]
  ...

ReID answers:
  Person A (CAM1) = Person Y (CAM2) = Person Q (CAM3)    ← Same person!
  Person B (CAM1) = Person X (CAM2) = Person P (CAM3)    ← Same person!
  Person C (CAM1) = not visible in CAM2 = Person R (CAM3)
```

This correspondence is what triangulation needs in Tutorial 3.

### The Steps

```
Video + Pose Data (all cameras)
       │
       ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  Crop People  │ ──► │  Extract      │ ──► │  Compare     │
│  from Frames  │     │  Embeddings   │     │  Per-Frame   │
└──────────────┘     └──────────────┘     └──────────────┘
  Bounding box         OSNet encoder        Cosine similarity
  or torso crop        512-dim vector       across cameras
                                                  │
                                                  ▼
                                           Identity Map
                                           (fed to Tutorial 3)
```

### Cropping Strategies

| Strategy | What | When to Use |
|---|---|---|
| **Full-body crop** | Entire person bounding box | General ReID |
| **Torso/shirt crop** | Shoulder-to-hip region | When lower body is occluded (desks, chairs) |

### Why Torso Crops?

In many scenarios (classrooms, crowded scenes), the lower body is occluded by desks/chairs.
Shirt color and pattern are often the most discriminative features, so focusing on the
torso region can actually **improve** matching accuracy.

---

## Part 3: Cropping People from Video

### How It Works

1. Load the video and the corresponding SLEAP H5 pose file
2. For each frame, read the tracked keypoints and compute bounding boxes
3. Crop each detected person from the frame
4. Resize to a standard size (256x256 for full body, 128x128 for torso)
5. Save crops with naming: `{camera}_{frame}_{instance}.jpg`

### H5 Pose Data Format (from Tutorial 1)

```
tracks shape: (n_instances, 2, n_nodes, n_frames)
                     │       │      │         │
                     │       │      │         └─ frame index
                     │       │      └─ 17 COCO keypoints
                     │       └─ x,y coordinates
                     └─ detected people
```

In [ ]:
# ============================================================
# STEP 3: Full-body person cropping from video + pose data
# ============================================================
import cv2
import h5py
import os
import numpy as np
from tqdm import tqdm

# ── CONFIGURE ──────────────────────────────────────────────
video_paths = [
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/aligned_CAM1_1_28_00_to_1_30_00.mp4",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/aligned_CAM2_1_28_00_to_1_30_00.mp4",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/aligned_CAM3_1_28_00_to_1_30_00.mp4",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/aligned_CAM4_1_28_00_to_1_30_00.mp4",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/aligned_CAM5_1_28_00_to_1_30_00.mp4",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/aligned_CAM6_1_28_00_to_1_30_00.mp4",
]

h5_paths = [
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/pose_results/aligned_CAM1_1_28_00_to_1_30_00_pose.000_aligned_CAM1_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/pose_results/aligned_CAM2_1_28_00_to_1_30_00_pose.000_aligned_CAM2_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/pose_results/aligned_CAM3_1_28_00_to_1_30_00_pose.000_aligned_CAM3_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/pose_results/aligned_CAM4_1_28_00_to_1_30_00_pose.000_aligned_CAM4_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/pose_results/aligned_CAM5_1_28_00_to_1_30_00_pose.000_aligned_CAM5_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_summer_camp_data/Videos_7_18_2025/pose_results/aligned_CAM6_1_28_00_to_1_30_00_pose.000_aligned_CAM6_1_28_00_to_1_30_00.analysis.h5",
]

out_dir = "reid_crops"
os.makedirs(out_dir, exist_ok=True)

CROP_SIZE = 256   # resize crops to this square size
PADDING = 40      # pixels of padding around bounding box
# ───────────────────────────────────────────────────────────

total_crops = 0

for video_path, h5_path in zip(video_paths, h5_paths):
    cam_name = os.path.basename(video_path).split("_")[1]  # e.g., "CAM1"
    print(f"\nProcessing {cam_name}...")
    
    # Load pose data from SLEAP H5
    with h5py.File(h5_path, 'r') as f:
        tracks = f['tracks'][()]  # shape: (n_instances, 2, n_nodes, n_frames)
    
    n_instances, _, n_nodes, n_frames = tracks.shape
    print(f"  Tracks shape: {tracks.shape}")
    print(f"  {n_instances} instances, {n_nodes} nodes, {n_frames} frames")
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ERROR: Could not open {video_path}")
        continue
    
    cam_crops = 0
    
    for frame_idx in tqdm(range(n_frames), desc=f"  {cam_name}"):
        ret, frame = cap.read()
        if not ret:
            break
        
        h, w = frame.shape[:2]
        
        for inst_idx in range(n_instances):
            # Get keypoints for this instance in this frame
            # tracks shape: (instances, xy, nodes, frames)
            x_coords = tracks[inst_idx, 0, :, frame_idx]  # all x for this instance
            y_coords = tracks[inst_idx, 1, :, frame_idx]  # all y for this instance
            
            # Skip if all NaN (person not visible)
            valid = ~np.isnan(x_coords) & ~np.isnan(y_coords)
            if not valid.any():
                continue
            
            # Compute bounding box from valid keypoints
            x_min = int(np.nanmin(x_coords[valid])) - PADDING
            x_max = int(np.nanmax(x_coords[valid])) + PADDING
            y_min = int(np.nanmin(y_coords[valid])) - PADDING
            y_max = int(np.nanmax(y_coords[valid])) + PADDING
            
            # Clamp to frame boundaries
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(w, x_max)
            y_max = min(h, y_max)
            
            # Skip tiny crops
            if (x_max - x_min) < 20 or (y_max - y_min) < 20:
                continue
            
            # Crop and resize
            crop = frame[y_min:y_max, x_min:x_max]
            crop_resized = cv2.resize(crop, (CROP_SIZE, CROP_SIZE))
            
            # Save: camera_frameNNNNN_instN.jpg
            fname = f"{cam_name}_frame{frame_idx:05d}_inst{inst_idx}.jpg"
            cv2.imwrite(os.path.join(out_dir, fname), crop_resized)
            cam_crops += 1
    
    cap.release()
    total_crops += cam_crops
    print(f"  Saved {cam_crops} crops")

print(f"\nTotal crops saved: {total_crops} in '{out_dir}/'")

---

## Part 4: Torso/Shirt Cropping

For more robust ReID (especially in classrooms or seated scenarios), we crop just the
**torso region** (shoulders to hips).

### Keypoints Used

```
   left_shoulder(5) ──── right_shoulder(6)
         |                      |
         |    TORSO REGION      |
         |                      |
     left_hip(11) ──────── right_hip(12)
```

We compute a bounding box from these 4 keypoints with some padding.

In [ ]:
# ============================================================
# STEP 4: Torso/shirt cropping using shoulder and hip keypoints
# ============================================================

out_dir_shirts = "shirt_crops"
os.makedirs(out_dir_shirts, exist_ok=True)

SHIRT_CROP_SIZE = 128

# COCO keypoint indices for the four torso corners
TORSO_INDICES = {
    "left_shoulder": 5,
    "right_shoulder": 6,
    "left_hip": 11,
    "right_hip": 12,
}

total_shirt_crops = 0

for video_path, h5_path in zip(video_paths, h5_paths):
    cam_name = os.path.basename(video_path).split("_")[1]
    print(f"\nProcessing {cam_name} (shirt crops)...")
    
    with h5py.File(h5_path, 'r') as f:
        tracks = f['tracks'][()]
    
    n_instances, _, n_nodes, n_frames = tracks.shape
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ERROR: Could not open {video_path}")
        continue
    
    cam_crops = 0
    
    for frame_idx in tqdm(range(n_frames), desc=f"  {cam_name}"):
        ret, frame = cap.read()
        if not ret:
            break
        
        h, w = frame.shape[:2]
        
        for inst_idx in range(n_instances):
            # Get torso keypoints
            torso_x = []
            torso_y = []
            for name, idx in TORSO_INDICES.items():
                x = tracks[inst_idx, 0, idx, frame_idx]
                y = tracks[inst_idx, 1, idx, frame_idx]
                if not np.isnan(x) and not np.isnan(y):
                    torso_x.append(x)
                    torso_y.append(y)
            
            # Need at least 2 torso keypoints for a useful crop
            if len(torso_x) < 2:
                continue
            
            # Compute torso bounding box with padding
            x_min = int(min(torso_x)) - 10
            x_max = int(max(torso_x)) + 10
            y_min = int(min(torso_y)) - 10
            y_max = int(max(torso_y)) + 10
            
            # Clamp to frame
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(w, x_max)
            y_max = min(h, y_max)
            
            if (x_max - x_min) < 10 or (y_max - y_min) < 10:
                continue
            
            crop = frame[y_min:y_max, x_min:x_max]
            crop_resized = cv2.resize(crop, (SHIRT_CROP_SIZE, SHIRT_CROP_SIZE))
            
            fname = f"{cam_name}_frame{frame_idx:05d}_inst{inst_idx}_shirt.jpg"
            cv2.imwrite(os.path.join(out_dir_shirts, fname), crop_resized)
            cam_crops += 1
    
    cap.release()
    total_shirt_crops += cam_crops
    print(f"  Saved {cam_crops} shirt crops")

print(f"\nTotal shirt crops: {total_shirt_crops} in '{out_dir_shirts}/'")

---

## Part 5: Extracting ReID Embeddings

### What is an Embedding?

An **embedding** is a fixed-length vector (512 dimensions) that encodes the visual
appearance of a person. The key property:

- **Same person** in different cameras → embeddings are **close** (high cosine similarity)
- **Different people** → embeddings are **far apart** (low cosine similarity)

### Using BoxMOT's StrongSort Encoder

BoxMOT's StrongSort tracker includes an OSNet-based ReID encoder. We use it directly:

1. Initialize the StrongSort tracker with ReID weights
2. Access the internal `encoder`
3. Pass crops through the encoder → 512-dim feature vectors
4. L2-normalize the features for cosine similarity comparison

In [ ]:
# ============================================================
# STEP 5: Initialize the ReID encoder
# ============================================================
import torch
import numpy as np
from pathlib import Path
from torchvision import transforms
from PIL import Image
from boxmot.trackers.strongsort.strongsort import StrongSort

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Path to pre-trained OSNet weights
reid_weights = Path("/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/osnet_x0_25_imagenet.pth")
assert reid_weights.exists(), f"ReID weights not found: {reid_weights}"

# Initialize StrongSort — this loads the OSNet encoder internally
tracker = StrongSort(
    reid_weights=reid_weights,
    device=device,
    half=False  # use full precision
)

# The encoder is accessible as an attribute
encoder = tracker.encoder
print(f"ReID encoder loaded successfully")
print(f"Weights: {reid_weights.name}")

In [ ]:
# ============================================================
# STEP 6: Define the image preprocessing pipeline
# ============================================================
# The OSNet model expects:
#   - Input size: 256 x 128 (height x width) — standard for person ReID
#   - Normalized with ImageNet mean/std
#   - Tensor format: (batch, channels, height, width)

transform = transforms.Compose([
    transforms.Resize((256, 128)),      # H, W — standard ReID size
    transforms.ToTensor(),               # [0,255] → [0,1], HWC → CHW
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],      # ImageNet means
        std=[0.229, 0.224, 0.225]        # ImageNet stds
    ),
])

# Quick test on a single crop
test_crops = [f for f in os.listdir("shirt_crops") if f.endswith(".jpg")][:1]
if test_crops:
    img = Image.open(os.path.join("shirt_crops", test_crops[0])).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    print(f"Input tensor shape:  {tensor.shape}  → (batch, C, H, W)")
    
    with torch.inference_mode():
        feat = encoder(tensor)
    print(f"Output embedding:    {feat.shape}  → 512-dimensional appearance vector")
else:
    print("No shirt crops found. Run Steps 3-4 first.")

In [ ]:
# ============================================================
# STEP 7: Extract embeddings from all crops (with sampling)
# ============================================================
from tqdm import tqdm

crop_dir = "shirt_crops"
sample_size = 1000  # sample subset for efficiency (use all for production)

# Gather all crop paths
all_crops = [os.path.join(crop_dir, f) for f in os.listdir(crop_dir) if f.endswith(".jpg")]
print(f"Found {len(all_crops)} total crops")

# Sample a subset for analysis
np.random.seed(42)
sample_crops = np.random.choice(all_crops, min(sample_size, len(all_crops)), replace=False)
print(f"Sampling {len(sample_crops)} crops for analysis")

# Extract embeddings
features = []
crop_names = []

with torch.inference_mode():
    for img_path in tqdm(sample_crops, desc="Extracting embeddings"):
        img = Image.open(img_path).convert("RGB")
        tensor = transform(img).unsqueeze(0).to(device)  # 1x3x256x128
        feat = encoder(tensor)                            # 1x512
        features.append(feat.cpu().numpy())
        crop_names.append(os.path.basename(img_path))

# Stack and L2-normalize
features = np.vstack(features)                            # (N, 512)
features = features / np.linalg.norm(features, axis=1, keepdims=True)

print(f"\nFeature matrix shape: {features.shape}")
print(f"  → {features.shape[0]} crops x {features.shape[1]} dimensions")
print(f"  → All vectors are L2-normalized (unit length)")

---

## Part 6: Computing Cosine Similarity

### What is Cosine Similarity?

Cosine similarity measures the angle between two vectors:

```
cosine_sim(A, B) = (A . B) / (||A|| x ||B||)
```

Since our features are L2-normalized (||A|| = ||B|| = 1), this simplifies to a dot product.

| Similarity | Interpretation |
|---|---|
| > 0.6 | Likely the same person |
| 0.3 - 0.6 | Uncertain — may need more evidence |
| < 0.3 | Definitely different people |

In [ ]:
# ============================================================
# STEP 8: Compute pairwise cosine similarity
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity

print("Computing cosine similarity matrix...")
sim_matrix = cosine_similarity(features)

print(f"Similarity matrix shape: {sim_matrix.shape}")
print(f"  → {sim_matrix.shape[0]} x {sim_matrix.shape[1]} pairwise comparisons")

# Statistics (exclude self-similarity on diagonal)
mask = ~np.eye(sim_matrix.shape[0], dtype=bool)
off_diag = sim_matrix[mask]

print(f"\nSimilarity statistics (off-diagonal):")
print(f"  Mean:   {off_diag.mean():.4f}")
print(f"  Std:    {off_diag.std():.4f}")
print(f"  Min:    {off_diag.min():.4f}")
print(f"  Max:    {off_diag.max():.4f}")
print(f"  Median: {np.median(off_diag):.4f}")

In [ ]:
# ============================================================
# STEP 9: Visualize the similarity matrix
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap of first 50 crops
ax1 = axes[0]
sns.heatmap(sim_matrix[:50, :50], cmap='RdYlBu_r', vmin=0, vmax=1,
            ax=ax1, cbar_kws={'label': 'Cosine Similarity'})
ax1.set_title('Pairwise Cosine Similarity (first 50 crops)')
ax1.set_xlabel('Crop Index')
ax1.set_ylabel('Crop Index')

# Histogram of all similarities
ax2 = axes[1]
ax2.hist(off_diag, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(x=0.6, color='red', linestyle='--', label='Match threshold (0.6)')
ax2.axvline(x=0.3, color='orange', linestyle='--', label='Reject threshold (0.3)')
ax2.set_xlabel('Cosine Similarity')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Pairwise Similarities')
ax2.legend()

plt.tight_layout()
plt.savefig("reid_similarity_analysis.png", dpi=150)
plt.show()
print("Saved: reid_similarity_analysis.png")

---

## Part 7: Per-Frame Cross-Camera Matching

This is the key step that produces the **identity map** needed by triangulation.

### Strategy

For each frame:
1. Gather the crops from all cameras for that frame
2. Extract embeddings for each crop
3. Compute cross-camera similarity: for each person in Cam A, find the best match in Cam B
4. Apply a threshold to filter false matches
5. Build a **correspondence table**: which instance in each camera maps to which global ID

### Output: Cross-Camera Identity Map

```
Frame 100:
  Global Person 0 → {CAM1: inst2, CAM2: inst0, CAM3: inst1, CAM5: inst3}
  Global Person 1 → {CAM1: inst0, CAM3: inst2, CAM6: inst0}
  ...
```

This is exactly what Tutorial 3 (triangulation) needs to know which 2D points to combine.

In [ ]:
# ============================================================
# STEP 10: Group crops by camera for cross-camera analysis
# ============================================================

def parse_crop_name(name):
    """Extract camera, frame, instance from crop filename.
    Format: CAM1_frame00100_inst0_shirt.jpg
    """
    parts = name.replace('.jpg', '').split('_')
    camera = parts[0]                              # e.g., "CAM1"
    frame = int(parts[1].replace('frame', ''))     # e.g., 100
    inst = int(parts[2].replace('inst', ''))       # e.g., 0
    return camera, frame, inst

# Group features by camera
camera_features = {}  # {camera: [(global_idx, frame, inst, feature), ...]}

for i, name in enumerate(crop_names):
    try:
        camera, frame, inst = parse_crop_name(name)
        if camera not in camera_features:
            camera_features[camera] = []
        camera_features[camera].append((i, frame, inst, features[i]))
    except (ValueError, IndexError):
        continue

print("Crops per camera:")
for cam, items in sorted(camera_features.items()):
    print(f"  {cam}: {len(items)} crops")

In [ ]:
# ============================================================
# STEP 11: Cross-camera matching — find same person across views
# ============================================================
MATCH_THRESHOLD = 0.5  # minimum similarity to consider a match

cameras = sorted(camera_features.keys())
print(f"Cameras: {cameras}")
print(f"Match threshold: {MATCH_THRESHOLD}\n")

# Compare every camera pair
for i, cam_a in enumerate(cameras):
    for cam_b in cameras[i+1:]:
        feats_a = np.array([item[3] for item in camera_features[cam_a]])
        feats_b = np.array([item[3] for item in camera_features[cam_b]])
        
        # Cross-camera similarity matrix
        cross_sim = cosine_similarity(feats_a, feats_b)
        
        # For each crop in cam_a, find its best match in cam_b
        best_matches = cross_sim.max(axis=1)
        best_indices = cross_sim.argmax(axis=1)
        
        # Filter by threshold
        good_matches = best_matches > MATCH_THRESHOLD
        n_good = good_matches.sum()
        
        print(f"{cam_a} <-> {cam_b}: "
              f"{n_good}/{len(best_matches)} matches above threshold "
              f"(avg sim: {best_matches.mean():.3f}, "
              f"max: {best_matches.max():.3f})")

In [ ]:
# ============================================================
# STEP 12: Visualize top matches between two cameras
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image

def show_top_matches(cam_a, cam_b, n_show=5):
    """Show the top N matching crop pairs between two cameras."""
    items_a = camera_features[cam_a]
    items_b = camera_features[cam_b]
    
    feats_a = np.array([item[3] for item in items_a])
    feats_b = np.array([item[3] for item in items_b])
    
    cross_sim = cosine_similarity(feats_a, feats_b)
    
    # Find top matches
    flat_indices = np.argsort(cross_sim.ravel())[::-1][:n_show]
    
    fig, axes = plt.subplots(n_show, 2, figsize=(6, 3 * n_show))
    if n_show == 1:
        axes = axes.reshape(1, 2)
    
    for rank, flat_idx in enumerate(flat_indices):
        idx_a = flat_idx // cross_sim.shape[1]
        idx_b = flat_idx % cross_sim.shape[1]
        sim = cross_sim[idx_a, idx_b]
        
        path_a = sample_crops[items_a[idx_a][0]]
        path_b = sample_crops[items_b[idx_b][0]]
        
        img_a = Image.open(path_a)
        img_b = Image.open(path_b)
        
        axes[rank, 0].imshow(img_a)
        axes[rank, 0].set_title(f"{cam_a} (sim={sim:.3f})")
        axes[rank, 0].axis('off')
        
        axes[rank, 1].imshow(img_b)
        axes[rank, 1].set_title(f"{cam_b}")
        axes[rank, 1].axis('off')
    
    plt.suptitle(f"Top {n_show} Matches: {cam_a} <-> {cam_b}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"matches_{cam_a}_{cam_b}.png", dpi=150)
    plt.show()

# Show matches between first two cameras
if len(cameras) >= 2:
    show_top_matches(cameras[0], cameras[1], n_show=5)

---

## Summary & Next Steps

### What We Accomplished

| Step | What | Output |
|---|---|---|
| Full-body cropping | Extracted person crops from all cameras | `reid_crops/` |
| Torso cropping | Focused shoulder-to-hip crops | `shirt_crops/` |
| ReID encoder | Loaded OSNet via BoxMOT StrongSort | 512-dim embeddings |
| Embedding extraction | Processed sampled crops | Feature matrix (N, 512) |
| Similarity analysis | Computed pairwise cosine similarity | Similarity matrix, histograms |
| Cross-camera matching | Identified same person across views per frame | Identity correspondence map |

### Why This Had to Come Before Triangulation

Without the identity map from this tutorial, triangulation would face an **ambiguity
problem**: if Camera 1 sees 5 people and Camera 3 sees 5 people, which person in
Camera 1 should be paired with which person in Camera 3? There are 5! = 120 possible
assignments — and the wrong one produces nonsensical 3D skeletons.

ReID gives us the correct assignment, so triangulation knows exactly which 2D keypoints
from different cameras to combine into each person's 3D skeleton.

### Next Tutorial

**Tutorial 3: 3D Triangulation** takes the 2D poses from Tutorial 1 along with the
cross-camera identity correspondences from this tutorial, and fuses them into 3D skeleton
coordinates using camera calibration and `sleap-anipose`.